In [0]:
%run ../00_common/data_utils

In [0]:
def is_not_blank(col):
    return col.isNotNull() & (F.trim(col) != "")

In [0]:
def calc_t_rtouchpoint_master_sap():

    sap_touchpoint_df = (spark.table(f"{get_env_config('silver_touchpoint_parsed_database')}.t_touchpoint_sapbi")
        .filter(is_not_blank(F.col("CustomerNumber")) & is_not_blank(F.col("DivisionCode")))
        .withColumn(
            "TouchPointTypeCode",
            F.when(F.col("BusinessType") == "Distributors", "distrib")
            .when(F.col("BusinessType") == "Perfumery/Pharmacy", "prfphr")
            .when(F.col("BusinessType") == "Specialty Multi", "smc")
            .when(F.col("BusinessType") == "Other", "other")
            .when(F.col("BusinessType") == "Free Standing", "frestnstr")
            .when(F.col("BusinessType") == "Corp Sales", "crpstr")
            .when(F.col("BusinessType") == "Salon/Spa", "spasln")
            .when(F.col("BusinessType") == "Online Pure Play", "onlpureply")
            .when(F.col("BusinessType") == "Department Stores", "dprstr")
            .when(F.col("BusinessType") == "Intercompany", "intrcmp")
            .when(F.col("BusinessType") == "Off Price", "offprcrtl")
            .when(F.col("BusinessType") == "Brand.com", "onl")
            .when(F.col("BusinessType") == "Online 3PP", "3pp")
            .when(F.col("BusinessType") == "Travel", "trvrtl")
            .when(F.col("BusinessType") == "Not assigned", "")
            .when(F.col("BusinessType") == "No Business Type", "")
            .when(F.col("BusinessType").isNull(), "")
            .otherwise(F.col("BusinessType"))
        )
        .withColumn("Channel", F.when((F.col("RetailerOnlineDoor") == "Y") | (F.col("TouchPointTypeCode") == "onl"),"Online").otherwise("Offline"))
        .withColumn("DTCFlag", F.when(F.col("TouchPointTypeCode").isin("frestnstr", "crpstr", "onl"), "true").otherwise("false"))
        .select(
            F.col("tp_sapbi_id"),
            F.col("CustomerNumber"),
            F.col("DivisionCode"),
            F.col("Channel"),
            F.col("TouchPointTypeCode"),
            F.col("CustomerGroup"),
            F.col("Retailer").alias("RetailerHierarchyCode"),  #需要和touchpoint master进行对比取值
            F.col("DTCFlag"),
            F.col("Region"),
            F.col("City"),
            F.lit("Global").alias("GLOBAL_Level"),
            F.col("ReportingGroup_Global_Code").alias("Global_Code"),
            F.col("ReportingGroup_Global_Name").alias("Global_Name"),
            F.lit("").alias("GLOBAL_DESCRIPTION"),
            F.lit("Regional").alias("REGIONAL_Level"),
            F.col("ReportingGroup_Regional_Code").alias("Regional_Code"),
            F.col("ReportingGroup_Regional_Name").alias("Regional_Name"),
            F.lit("").alias("REGIONAL_DESCRIPTION"),
            F.lit("Affiliate").alias("AFFILIATE_Level"),
            F.col("ReportingGroup_Affiliate_Code").alias("Affiliate_Code"),
            F.col("ReportingGroup_Affiliate_Name").alias("Affiliate_Name"),
            F.lit("").alias("AFFILIATE_DESCRIPTION")
        )
        .fillna("")
    )

    # 定义排序字段
    # order_cols = [
    #     F.col("MarketCode").desc_nulls_last(),
    #     F.col("SalesOrganisation").desc_nulls_last(),
    #     F.col("CustomerName").desc_nulls_last(),
    #     F.col("BusinessType").desc_nulls_last(),
    #     F.col("RetailerOnlineDoor").desc_nulls_last(),
    #     F.col("CustomerGroup").desc_nulls_last(),
    #     F.col("ReportingGroup_Global_Code").desc_nulls_last(),
    #     F.col("ReportingGroup_Global_Name").desc_nulls_last(),
    #     F.col("ReportingGroup_Regional_Code").desc_nulls_last(),
    #     F.col("ReportingGroup_Regional_Name").desc_nulls_last(),
    #     F.col("ReportingGroup_Affiliate_Code").desc_nulls_last(),
    #     F.col("ReportingGroup_Affiliate_Name").desc_nulls_last(),
    #     F.col("Retailer").desc_nulls_last(),
    #     F.col("Region").desc_nulls_last(),
    #     F.col("City").desc_nulls_last(),
    #     F.col("CREATION_DT").desc_nulls_last(),
    #     F.col("UPDATE_DT").desc_nulls_last()
    # ]

    # 定义窗口：按 CustomerNumber + DivisionCode 分区，按上述字段降序排序
    window_spec = Window.partitionBy("CustomerNumber", "DivisionCode").orderBy(F.col("tp_sapbi_id"))

    # 应用 row_number（或其他聚合）
    sap_touchpoint_deduped = (
        sap_touchpoint_df
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )


    touchpoint_master_df = (spark.table(f"{get_env_config('golden_touchpoint_master_database')}.t_touchpoint_master")
        .filter(F.isnotnull("TCPM_MarketCode") & F.isnotnull("TCPM_BrandCode") & F.isnotnull("TCPM_TouchPointCode"))
        .withColumn("rn", F.row_number().over(Window.partitionBy("TCPM_MarketCode", "TCPM_BrandCode", "TCPM_TouchPointCode")
                                              .orderBy(
                                                  F.col("TCPM_SourceTimestamp").desc(),
                                                  F.col("KAFKA_TIMESTAMP").desc(),
                                                  F.col("TCPM_UPDATE_DT").desc(),
                                                  F.col("TCPM_ID").desc()
                                              )
                    )
        )
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    result_df = (touchpoint_master_df.alias("a")
        .join(sap_touchpoint_deduped.alias("b"),
              (F.col("a.TCPM_BrandCode") == F.col("b.DivisionCode")) & (F.col("a.TCPM_CustomerNumber") == F.col("b.CustomerNumber")),
              "left"
        )
        .select(
            F.col("a.TCPM_ID").alias("ID"),
            F.col("a.TCPM_TCPT_ID").alias("TCPT_ID"),
            F.col("b.TP_SAPBI_ID").alias("TP_SAPBI_ID"),
            F.col("a.tcpm_action").alias("action"),
            F.col("a.tcpm_documenttimestamp").alias("documenttimestamp"),
            F.col("a.tcpm_documentuuid").alias("documentuuid"),
            F.col("a.tcpm_recorduuid").alias("recorduuid"),
            F.col("a.TCPM_SourceSystemCode").alias("SourceSystemCode"),
            F.col("a.TCPM_SourceTimestamp").alias("SourceTimestamp"),
            F.col("a.TCPM_MarketCode").alias("MarketCode"),
            F.col("a.TCPM_AffiliateCode").alias("AffiliateCode"),
            F.col("a.TCPM_DivisionCode").alias("DivisionCode"),
            F.col("a.TCPM_BrandCode").alias("BrandCode"),
            F.col("a.TCPM_TouchPointCode").alias("TouchPointCode"),
            F.col("a.TCPM_SubMarketCode").alias("SubMarketCode"),
            F.col("a.TCPM_AuxiliaryCode").alias("AuxiliaryCode"),
            F.col("a.TCPM_AuxiliaryTouchPointCode").alias("AuxiliaryTouchPointCode"),
            F.col("a.TCPM_HygieneServiceCode").alias("HygieneServiceCode"),
            F.col("a.TCPM_DistributionChannelCode").alias("DistributionChannelCode"),
            F.col("a.TCPM_TouchPointGroupCode").alias("TouchPointGroupCode"),
            F.col("a.TCPM_EnglishDescription").alias("EnglishDescription"),
            F.col("a.TCPM_LocalDescription").alias("LocalDescription"),
            F.col("a.TCPM_EnglishFullDescription").alias("EnglishFullDescription"),
            F.col("a.TCPM_LocalFullDescription").alias("LocalFullDescription"),
            F.col("a.TCPM_Descriptionen").alias("Descriptionen"),
            F.col("a.TCPM_Descriptionlocal").alias("Descriptionlocal"),
            F.col("a.TCPM_FullDescriptionen").alias("FullDescriptionen"),
            F.col("a.TCPM_FullDescriptionlocal").alias("FullDescriptionlocal"),
            F.col("a.TCPM_URL").alias("URL"),
            F.col("a.TCPM_JDECode").alias("JDECode"),
            F.col("a.TCPM_Active").alias("Active"),
            F.col("a.TCPM_TouchPointStatus").alias("TouchPointStatus"),
            F.col("a.TCPM_OpenDate").alias("OpenDate"),
            F.col("a.TCPM_BranchID").alias("BranchID"),
            F.col("a.TCPM_CustomerNumber").alias("CustomerNumber"),
            F.col("a.TCPM_RedirectTouchPointCode").alias("RedirectTouchPointCode"),
            F.col("a.TCPM_PHONELIST").alias("PHONELIST"),
            F.col("a.TCPM_ADDRESSLIST").alias("ADDRESSLIST"),
            F.col("a.TCPM_CUSTOMATTRIBUTELIST").alias("CUSTOMATTRIBUTELIST"),
            F.col("a.TCPM_TERMINALREGISTRATIONLIST").alias("TERMINALREGISTRATIONLIST"),
            F.col("a.TCPM_Attr_CustomAttributeList").alias("Attr_CustomAttributeList"),
            F.col("a.TCPM_CREATIONUID").alias("CREATIONUID"),
            F.col("a.TCPM_UPDATEUID").alias("UPDATEUID"),
            F.current_timestamp().alias("creation_dt"),
            F.current_timestamp().alias("update_dt"),

            F.when(F.coalesce(F.col("b.Channel"), F.lit("")) != F.lit(""), F.col("b.Channel")).otherwise(F.col("a.tcpm_channel")).alias("Channel"),
            F.when(F.coalesce(F.col("b.TouchPointTypeCode"), F.lit("")) != F.lit(""), F.col("b.TouchPointTypeCode")).otherwise(F.col("a.tcpm_touchpointtypecode")).alias("TouchPointTypeCode"),
            F.when(F.coalesce(F.col("b.CustomerGroup"), F.lit("")) != F.lit(""), F.col("b.CustomerGroup")).otherwise(F.col("a.TCPM_CustomerGroup")).alias("CustomerGroup"),
            F.when(F.coalesce(F.col("b.RetailerHierarchyCode"), F.lit("")) != F.lit(""), F.col("b.RetailerHierarchyCode")).otherwise(F.col("a.tcpm_retailerhierarchycode")).alias("RetailerHierarchyCode"),
            F.when(F.coalesce(F.col("b.DTCFlag"), F.lit("")) != F.lit(""), F.col("b.DTCFlag")).otherwise(F.col("a.tcpm_dtcflag")).alias("DTCFlag"),
            F.when(F.coalesce(F.col("b.Region"), F.lit("")) != F.lit(""), F.col("b.Region")).otherwise(F.col("a.TCPM_Region")).alias("Region"),
            F.when(F.coalesce(F.col("b.City"), F.lit("")) != F.lit(""), F.col("b.City")).otherwise(F.col("a.TCPM_City")).alias("City"),

            F.when(F.coalesce(F.col("b.GLOBAL_Level"), F.lit("")) != F.lit(""), F.col("b.GLOBAL_Level")).otherwise(F.col("a.TCPM_GLOBAL_Level")).alias("GLOBAL_Level"),
            F.when(F.coalesce(F.col("b.Global_Code"), F.lit("")) != F.lit(""), F.col("b.Global_Code")).otherwise(F.col("a.tcpm_global_code")).alias("Global_Code"),
            F.when(F.coalesce(F.col("b.Global_Name"), F.lit("")) != F.lit(""), F.col("b.Global_Name")).otherwise(F.col("a.tcpm_global_name")).alias("Global_Name"),
            F.when(F.coalesce(F.col("b.GLOBAL_DESCRIPTION"), F.lit("")) != F.lit(""), F.col("b.GLOBAL_DESCRIPTION")).otherwise(F.col("a.tcpm_global_description")).alias("GLOBAL_DESCRIPTION"),

            F.when(F.coalesce(F.col("b.REGIONAL_Level"), F.lit("")) != F.lit(""), F.col("b.REGIONAL_Level")).otherwise(F.col("a.TCPM_REGIONAL_Level")).alias("REGIONAL_Level"),
            F.when(F.coalesce(F.col("b.Regional_Code"), F.lit("")) != F.lit(""), F.col("b.Regional_Code")).otherwise(F.col("a.tcpm_regional_code")).alias("Regional_Code"),
            F.when(F.coalesce(F.col("b.Regional_Name"), F.lit("")) != F.lit(""), F.col("b.Regional_Name")).otherwise(F.col("a.tcpm_regional_name")).alias("Regional_Name"),
            F.when(F.coalesce(F.col("b.REGIONAL_DESCRIPTION"), F.lit("")) != F.lit(""), F.col("b.REGIONAL_DESCRIPTION")).otherwise(F.col("a.tcpm_regional_description")).alias("REGIONAL_DESCRIPTION"),

            F.when(F.coalesce(F.col("b.AFFILIATE_Level"), F.lit("")) != F.lit(""), F.col("b.AFFILIATE_Level")).otherwise(F.col("a.TCPM_AFFILIATE_Level")).alias("AFFILIATE_Level"),
            F.when(F.coalesce(F.col("b.Affiliate_Code"), F.lit("")) != F.lit(""), F.col("b.Affiliate_Code")).otherwise(F.col("a.tcpm_affiliate_code")).alias("Affiliate_Code"),
            F.when(F.coalesce(F.col("b.Affiliate_Name"), F.lit("")) != F.lit(""), F.col("b.Affiliate_Name")).otherwise(F.col("a.tcpm_affiliate_name")).alias("Affiliate_Name"),
            F.when(F.coalesce(F.col("b.AFFILIATE_DESCRIPTION"), F.lit("")) != F.lit(""), F.col("b.AFFILIATE_DESCRIPTION")).otherwise(F.col("a.tcpm_affiliate_description")).alias("AFFILIATE_DESCRIPTION"),

            F.col("a.KAFKA_TIMESTAMP").alias("KAFKA_TIMESTAMP")
        )
    )

    save_to_target_table(result_df, f"{get_env_config('golden_touchpoint_master_database')}.t_touchpoint_master_sap")
    

In [0]:
batch_id = dbutils.widgets.get("batch_id")
print(f"batch_id: {batch_id}")

with StepLogger("t_touchpoint_master_sap", "05-2", "touchpoint", task_id=batch_id) as logger:
    calc_t_rtouchpoint_master_sap()